In [ ]:
import pandas as pd
from google.colab import drive
from pathlib import Path
import matplotlib.pyplot as plt


In [ ]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [ ]:
drive.mount('/content/drive')

In [ ]:
path = Path("/content/drive/MyDrive/Colab Notebooks/swiss_news_with_category.csv")

In [ ]:
df = pd.read_csv(path)

In [ ]:
df.head()

In [ ]:
df = df[ df['topic_category'] == "financial"]

In [ ]:
assert {"company", "head_clean", "content_clean", "pubtime"}.issubset(df.columns), \
    "Expected columns company/head_clean/content_clean/pubtime are missing."

# Ensure 'pubtime' is converted to datetime, handle errors gracefully
df["pubtime"] = pd.to_datetime(df["pubtime"], errors="coerce")

# Combine headline + a short snippet of body text
SNIPPET_LEN = 500  # Set as reasonable default if not set elsewhere

def build_text(row):
    head = str(row.get("head_clean", "")).strip()
    body = str(row.get("content_clean", "")).strip()
    return (head + " " + body[:SNIPPET_LEN]).strip()

tmp = pd.DataFrame({
    "date":     df["pubtime"].dt.date,  # Will be NaT for unparsable dates
    "ticker":   df["company"].astype(str),
    "language": df["language"].astype(str),
    "text":     df.apply(build_text, axis=1).astype(str)
})

# Drop rows with missing date, ticker, or text, or very short texts
tmp = tmp.dropna(subset=["date", "ticker", "text"])
tmp = tmp[tmp["text"].str.len() > 3].copy()

print("Rows after cleaning:", len(tmp))
tmp.head(3)



In [ ]:
# drop unparsable dates if any slipped through
tmp = tmp.dropna(subset=["date"])

# normalize ticker format
tmp["ticker"] = tmp["ticker"].str.strip().str.upper()


In [ ]:
print("Rows after cleaning:", len(tmp))

In [ ]:
# --- Chunk 3: language distribution & strategy ---
lang_counts = tmp["language"].str.lower().value_counts()
display(lang_counts.head(10))

# we know your set is mixed; force multilingual to be explicit
MODEL_STRATEGY = "multilingual"
print("Model strategy:", MODEL_STRATEGY)


## XLM Roberta

In [ ]:
# --- Chunk 4: load multilingual model + define scorer ---

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
import numpy as np, torch

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"  # multilingual

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
mdl = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device).eval()

def score_texts(texts, max_len=256, batch_size=64):
    """
    texts: list[str]
    returns: np.ndarray of shape (len(texts),) with scores in [-1, 1]
             score = P(positive) - P(negative)
    """
    out = np.zeros(len(texts), dtype="float32")
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            enc = tok(
                batch,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=max_len
            ).to(device)
            logits = mdl(**enc).logits.detach().cpu().numpy()
            probs = softmax(logits, axis=1)   # order: [neg, neu, pos]
            out[i:i+batch_size] = probs[:,2] - probs[:,0]
    return out




In [ ]:
import math
# --- Chunk 5: score all articles ---

# optional: progress bar (comment out if you don't have tqdm)
try:
    from tqdm.auto import tqdm
    use_tqdm = True
except Exception:
    use_tqdm = False

texts = tmp["text"].tolist()
N = len(texts)
scores = np.zeros(N, dtype="float32")

batch = 64  # you can raise to 128 if you have plenty of GPU RAM
max_len = 256

mdl.eval()
with torch.no_grad():
    it = range(0, N, batch)
    if use_tqdm: it = tqdm(it, total=math.ceil(N/batch), desc="Scoring")
    for i in it:
        batch_texts = texts[i:i+batch]
        enc = tok(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_len
        ).to(device)
        logits = mdl(**enc).logits.detach().cpu().numpy()
        probs = softmax(logits, axis=1)   # [neg, neu, pos]
        scores[i:i+batch] = probs[:, 2] - probs[:, 0]  # pos - neg in [-1, 1]

tmp["sent_score"] = scores

# quick sanity check
tmp[["date","ticker","language","sent_score"]].head(5)
tmp["sent_score"].describe()


## German FinBERT

In [ ]:
# --- Chunk 5B: Add German FinBERT sentiment ---

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
import numpy as np, torch

device = "cuda" if torch.cuda.is_available() else "cpu"
FINBERT_DE_MODEL = "oliverguhr/german-sentiment-bert"

print("Loading German FinBERT model…")
tok_de = AutoTokenizer.from_pretrained(FINBERT_DE_MODEL, use_fast=False)
mdl_de = AutoModelForSequenceClassification.from_pretrained(FINBERT_DE_MODEL).to(device).eval()

def score_texts_de(texts, max_len=256, batch_size=64):
    """German FinBERT scoring"""
    out = np.zeros(len(texts), dtype="float32")
    mdl_de.eval()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            enc = tok_de(
                texts[i:i+batch_size],
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=max_len
            ).to(device)
            logits = mdl_de(**enc).logits.detach().cpu().numpy()
            probs = softmax(logits, axis=1)   # [neg, neu, pos]
            out[i:i+batch_size] = probs[:, 2] - probs[:, 0]
    return out

# score only German rows
mask_de = tmp["language"].str.lower().eq("de")
texts_de = tmp.loc[mask_de, "text"].tolist()
print(f"Scoring {len(texts_de):,} German articles with FinBERT…")

scores_de = score_texts_de(texts_de)
tmp.loc[mask_de, "sent_score_finbert"] = scores_de

# other languages: keep NaN (no FinBERT)
tmp["sent_score_finbert"] = tmp["sent_score_finbert"].fillna(np.nan)

# rename old XLM-R column for clarity
tmp = tmp.rename(columns={"sent_score": "sent_score_xlm"})

print("Done. New columns:")
print(tmp[["language", "sent_score_xlm", "sent_score_finbert"]].head(5))


In [ ]:
tmp.head(20)

In [ ]:
path = Path("/content/drive/MyDrive/Colab Notebooks/scored_categorized_news_data.csv")

In [ ]:
tmp = pd.read_csv(path)
tmp.head()

In [ ]:
tmp = tmp.dropna(subset=["sent_cat_finbert"])
tmp.head()

In [ ]:
tmp.shape

In [ ]:
tmp = tmp.drop(columns="sent_score_xlm")

In [ ]:
tmp.shape

In [ ]:
# 1) Work on a copy to be safe
tmp = tmp.copy()

# 2) Keep only German articles and non-null FinBERT scores
tmp = tmp[tmp["language"] == "de"].copy()
tmp = tmp[~tmp["sent_score_finbert"].isna()].copy()

# 3) Apply Denis' thresholds to create a fresh categorical label
def cat_from_score(x):
    if x <= -0.5:
        return "negative"
    elif x >= 0.5:
        return "positive"
    else:
        return "neutral"

tmp["sent_cat_finbert"] = tmp["sent_score_finbert"].apply(cat_from_score)

# 4) Quick check: how many of each category?
tmp["sent_cat_finbert"].value_counts()


In [ ]:
daily_sent = (
    tmp.groupby(["ticker", "date"])
       .agg(
           sent_mean=("sent_score_finbert", "mean"),
           sent_std=("sent_score_finbert", "std"),
           sent_count=("sent_score_finbert", "count"),
           sent_pos=("sent_cat_finbert", lambda x: (x == "positive").sum()),
           sent_neg=("sent_cat_finbert", lambda x: (x == "negative").sum()),
           sent_neu=("sent_cat_finbert", lambda x: (x == "neutral").sum()),
       )
       .reset_index()
)

# If only 1 article that day → std is NaN → set to 0
daily_sent["sent_std"] = daily_sent["sent_std"].fillna(0.0)

daily_sent.head()




In [ ]:
daily_sent = daily_sent.sort_values(["ticker", "date"])

daily_sent["sent_lag1"] = (
    daily_sent.groupby("ticker")["sent_mean"].shift(1)
)

daily_sent["sent_roll3"] = (
    daily_sent.groupby("ticker")["sent_mean"]
              .shift(1)  # don't include today's value
              .rolling(3, min_periods=1)
              .mean()
)


In [ ]:
# Ensure correct order
daily_sent = daily_sent.sort_values(["ticker", "date"]).reset_index(drop=True)

# 1) Yesterday's sentiment
daily_sent["sent_lag1"] = (
    daily_sent.groupby("ticker")["sent_mean"].shift(1)
)

# 2) 3-day rolling mean SENTIMENT UP TO YESTERDAY (no leakage)
daily_sent["sent_roll3"] = (
    daily_sent.groupby("ticker")["sent_mean"]
              .shift(1)                # exclude today's sentiment
              .rolling(3, min_periods=1)
              .mean()
)


In [ ]:
daily_sent.head(10)


In [ ]:
abb = daily_sent[daily_sent["ticker"]=="ABB"]

plt.figure(figsize=(10,4))
plt.plot(abb["date"], abb["sent_mean"], marker="o")
plt.title("ABB – Daily Mean Sentiment (FinBERT)")
plt.xlabel("Date")
plt.ylabel("Mean Sentiment")
plt.tight_layout()
plt.show()


In [ ]:
def pick_daily_category(row):
    if row["sent_pos"] > row["sent_neg"]:
        return "positive"
    elif row["sent_neg"] > row["sent_pos"]:
        return "negative"
    else:
        return "neutral"

daily_sent["sent_daily_cat"] = daily_sent.apply(pick_daily_category, axis=1)

daily_sent.head(10)


In [ ]:
daily_sent.to_parquet("/content/drive/MyDrive/Colab Notebooks/sentiment_daily.parquet", index=False)


In [ ]:
daily_sent.to_csv("/content/drive/MyDrive/Colab Notebooks/sentiment_daily.csv", index=False)

In [ ]:
import nbformat as nbf
import nbformat as nbf

path = "06_sentiment_categorized_news.ipynb"   # adjust if needed

nb = nbf.read(path, as_version=4)

# Remove ONLY the broken widgets metadata (keep everything else)
if "widgets" in nb.metadata:
    print("Removing notebook-level widget metadata...")
    del nb.metadata["widgets"]

# Remove widget metadata from cells but KEEP outputs
for cell in nb.cells:
    if "metadata" in cell and "widgets" in cell["metadata"]:
        print("Removing cell widget metadata...")
        del cell["metadata"]["widgets"]

# Save file back
nbf.write(nb, path)

print("Notebook successfully cleaned without removing outputs.")
